In [ ]:
import os
import torch
import torchaudio
from torchaudio.transforms import Resample
from datasets import Dataset, Audio, concatenate_datasets
from tqdm.auto import tqdm
import pandas as pd
from silero_vad import load_silero_vad

def load_audio_from_url(audio_url):
    """
    Load audio from a presigned URL of an MP3 file using torchaudio.
    """
    try:
        waveform, sample_rate = torchaudio.load(audio_url, format='mp3')
        return waveform, sample_rate
    except Exception as e:
        print(f"Error loading audio from {audio_url}: {e}")
        return None, None

def split_audio_into_chunks(waveform, sample_rate, max_chunk_duration=30, min_chunk_duration=5):
    """
    Use Silero VAD to split the audio into speech chunks.
    """
    # Ensure audio is mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    vad_model = load_silero_vad()

    # Get speech timestamps
    speech_timestamps = get_speech_ts(waveform.squeeze(), vad_model, sampling_rate=sample_rate)

    # Split into chunks based on VAD timestamps and merge to meet duration constraints
    chunks = []
    current_chunk = torch.empty(0, dtype=waveform.dtype)
    current_duration = 0.0

    for timestamp in speech_timestamps:
        start_sample = timestamp['start']
        end_sample = timestamp['end']
        speech_segment = waveform[:, start_sample:end_sample]
        speech_duration = (end_sample - start_sample) / sample_rate

        if speech_duration > max_chunk_duration:
            # Split long speech segment into smaller chunks
            num_subsegments = int(speech_duration // max_chunk_duration) + 1
            samples_per_subsegment = int((end_sample - start_sample) / num_subsegments)
            for i in range(num_subsegments):
                sub_start = start_sample + i * samples_per_subsegment
                sub_end = min(sub_start + samples_per_subsegment, end_sample)
                sub_segment = waveform[:, sub_start:sub_end]
                sub_duration = (sub_end - sub_start) / sample_rate
                if sub_duration >= min_chunk_duration:
                    chunks.append(sub_segment)
        else:
            if current_duration + speech_duration > max_chunk_duration:
                if current_duration >= min_chunk_duration:
                    chunks.append(current_chunk)
                    current_chunk = speech_segment
                    current_duration = speech_duration
                else:
                    # Append to current chunk even if it exceeds max_chunk_duration
                    current_chunk = torch.cat((current_chunk, speech_segment), dim=1)
                    current_duration += speech_duration
                    chunks.append(current_chunk)
                    current_chunk = torch.empty(0, dtype=waveform.dtype)
                    current_duration = 0.0
            else:
                current_chunk = torch.cat((current_chunk, speech_segment), dim=1)
                current_duration += speech_duration

    # Add any remaining chunk
    if current_duration >= min_chunk_duration:
        chunks.append(current_chunk)

    return chunks

def create_dataset_from_chunks(chunks, audio_id, sample_rate):
    """
    Create a Hugging Face Dataset from audio chunks.
    """
    data = []
    for idx, chunk in enumerate(chunks):
        # Convert chunk to NumPy array
        chunk_array = chunk.squeeze().numpy()
        data.append({
            'audio': {
                'array': chunk_array,
                'sampling_rate': sample_rate
            },
            'audio_id': f"{audio_id}_{idx+1}",
            'transcription': ''  # Empty transcription
        })

    dataset = Dataset.from_dict(data)
    # Cast 'audio' column to Audio feature
    dataset = dataset.cast_column('audio', Audio())
    return dataset

def process_and_save_datasets(csv_file, batch_size=1000, output_dir='./datasets', target_sample_rate=16000, max_audios=None):
    """
    Process audio URLs from a CSV file and save datasets in batches.
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Read the CSV file
    df = pd.read_csv(csv_file)
    audio_urls = df['audio_url'].tolist()
    audio_ids = df['audio_id'].tolist() if 'audio_id' in df.columns else [f"audio_{i+1}" for i in range(len(df))]

    # Limit to max_audios if specified
    if max_audios is not None:
        audio_urls = audio_urls[:max_audios]
        audio_ids = audio_ids[:max_audios]

    total_chunks = 0
    batch_datasets = []
    batch_count = 1

    for idx, (audio_url, audio_id) in enumerate(tqdm(zip(audio_urls, audio_ids), total=len(audio_urls), desc="Processing Audios")):
        waveform, sample_rate = load_audio_from_url(audio_url)
        if waveform is None:
            continue  # Skip if audio could not be loaded

        # Resample to target sample rate if necessary
        if sample_rate != target_sample_rate:
            resampler = Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
            waveform = resampler(waveform)
            sample_rate = target_sample_rate

        # Split audio into chunks
        chunks = split_audio_into_chunks(waveform, sample_rate)
        total_chunks += len(chunks)

        # Create dataset from chunks
        dataset = create_dataset_from_chunks(chunks, audio_id, sample_rate)
        batch_datasets.append(dataset)

        # Save dataset if batch size reached or last audio processed
        if total_chunks >= batch_size or idx == len(audio_urls) - 1:
            # Concatenate datasets in the batch
            if len(batch_datasets) > 1:
                batch_dataset = concatenate_datasets(batch_datasets)
            else:
                batch_dataset = batch_datasets[0]

            # Save dataset to disk
            dataset_path = os.path.join(output_dir, f"dataset_batch_{batch_count}")
            batch_dataset.save_to_disk(dataset_path)
            print(f"Saved dataset batch {batch_count} with {total_chunks} chunks to {dataset_path}")

            # Reset batch variables
            batch_datasets = []
            total_chunks = 0
            batch_count += 1

# Example usage
csv_file = 'full_audios_mp3_with_signed_urls.csv'  # Replace with your CSV file path
output_dir = './datasets'    # Directory to save datasets
batch_size = 1000            # Number of chunks per dataset
max_audios = 10              # Process only the first 10 audios

process_and_save_datasets(csv_file, batch_size=batch_size, output_dir=output_dir, max_audios=max_audios)

In [2]:
import pandas as pd
df = pd.read_csv('/Users/solanotodes/Documents/audios-trainer/datasets/full_audios_mp3_with_signed_urls.csv')

In [8]:
import soundfile as sf
import io
from urllib.request import urlopen

test_url = df.iloc[0]['audio_url']
data, samplerate = sf.read(io.BytesIO(urlopen(test_url).read()))

TypeError: Not allowed for existing files (except 'RAW'): samplerate, channels, format, subtype, endian